In [1]:
# Imports / global contants

# csv Dateien sind im Verzeichnis ../data zu finden

import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import math
import numpy as np
import seaborn as sns

from pandas.plotting import parallel_coordinates

timeFormat = "%Y-%m-%dT%H:%M:%S.%fZ"

export = "../export/"
export_img = "../export/img/"

path = "../data/"

percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]

In [2]:
df_study = pd.read_csv(rf'{export}data_complete.csv', sep= ";")
display(df_study)

,Unnamed: 0,DateTime,State,mappingMethod,TaskNo,TargetLayers,LayerCount,TrialIdx,posX,posY,posZ,TimeStamp,iteractionType,currentLayer,Proband,FrameTime
0,0,2021-06-29T07:13:59.622Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,NaN
1,1,2021-06-29T07:13:59.630Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,8.0
2,2,2021-06-29T07:13:59.673Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,43.0
3,3,2021-06-29T07:13:59.707Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,34.0
4,4,2021-06-29T07:13:59.742Z,INTERACTION,direct,1,"1,5",6,0,-,-,-,-,-,-,23,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2733075,2733075,2021-05-31T10:19:26.341Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,32.0
2733076,2733076,2021-05-31T10:19:26.416Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,75.0
2733077,2733077,2021-05-31T10:19:26.444Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,28.0
2733078,2733078,2021-05-31T10:19:26.474Z,INTERACTION,direct,1,"1,8,11,4,14",15,0,-,-,-,-,-,-,12,30.0


In [3]:
df_cleaned = pd.read_csv(rf'{export}cleanedStudy_interactionOnly_removed-inactive-300.csv', sep= ";")
display(df_cleaned)

,DateTime,Unnamed: 0,SampleIdx_global,SampleIdx_Proband,State,mappingMethod,Task,TargetLayers,NumLayers,Trial,...,iteractionType,currentLayer,Proband,FrameTime,shifted,Result,Block,Target,Target_Relative,DateTime2
0,2021-06-29 07:19:27.465000+00:00,0,65,65,VIEW,direct,1,"['3', '11', '9', '6', '1']",12,0,...,NaN,NaN,23,4.0,1.0,START,1,3,0.250000,2021-06-29 07:19:27.465000+00:00
1,2021-06-29 07:19:27.498000+00:00,1,66,66,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,33.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.498000+00:00
2,2021-06-29 07:19:27.540000+00:00,2,67,67,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,1,23,42.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.540000+00:00
3,2021-06-29 07:19:27.598000+00:00,3,68,68,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,58.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.598000+00:00
4,2021-06-29 07:19:27.637000+00:00,4,69,69,INTERACTION,direct,1,"['3', '11', '9', '6', '1']",12,0,...,-,-,23,39.0,1.0,COMPLETED,1,3,0.250000,2021-06-29 07:19:27.637000+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398926,2021-05-31 10:18:49.509000+00:00,1535230,2360990,90024,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,32.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.509000+00:00
1398927,2021-05-31 10:18:49.544000+00:00,1535231,2360991,90025,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,35.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.544000+00:00
1398928,2021-05-31 10:18:49.571000+00:00,1535232,2360992,90026,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,27.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.571000+00:00
1398929,2021-05-31 10:18:49.602000+00:00,1535233,2360993,90027,INTERACTION,widening,18,"['8', '11', '14', '4', '1']",15,4,...,1,1,12,31.0,54.0,COMPLETED,3,1,0.066667,2021-05-31 10:18:49.602000+00:00


In [4]:
def computeCI(df, m= 'mean', c='count', s='std'):
    df['ci95_hi'] = df[m] + 1.96*df[s]/(df[c].apply(np.sqrt))
    df['ci95_lo'] = df[m] - 1.96*df[s]/(df[c].apply(np.sqrt))

def computeWhiskers(df, desc, col_names, q1='25%', q3='75%'):
    desc['iqr'] = desc[q3] - desc[q1]

    # Whisker-Berechnung (innerhalb 1.5 * IQR)
    desc['lower_bound'] = desc[q1] - 1.5 * desc['iqr']
    desc['upper_bound'] = desc[q3] + 1.5 * desc['iqr']    

    lower_whiskers = []
    upper_whiskers = []

    for col_name in col_names:
        lower_bound = desc['lower_bound'].T[col_name]
        upper_bound = desc['upper_bound'].T[col_name]

        # Whiskers sind die letzten Punkte innerhalb dieser Grenzen
        l = df[df[col_name] >= lower_bound][col_name].min()
        u = df[df[col_name] <= upper_bound][col_name].max()

        lower_whiskers.append(l)
        upper_whiskers.append(u)

    desc['lower_whisker'] = lower_whiskers
    desc['upper_whisker'] = upper_whiskers

In [5]:
df_study['FrameTime'] = pd.to_datetime(df_study["DateTime"], format=timeFormat).diff().dt.microseconds / 1000.0

In [6]:
desc_complete = pd.DataFrame(df_study['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_complete)

computeWhiskers(df_study, desc_complete, ['FrameTime'])

display(desc_complete.T)

desc_complete.T.to_csv(rf'{export}desc-stats_framestats_complete.csv', sep=';')


,FrameTime
count,2.733079e+06
mean,3.750637e+01
std,8.842094e+00
min,0.000000e+00
5%,2.700000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.900000e+01


In [7]:
desc_interaction = pd.DataFrame(df_cleaned['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_interaction)

computeWhiskers(df_cleaned, desc_interaction, ['FrameTime'])

display(desc_interaction.T)

desc_interaction.T.to_csv(rf'{export}desc-stats_framestats_interaction.csv', sep=';')

,FrameTime
count,1.398931e+06
mean,3.742710e+01
std,8.674817e+00
min,0.000000e+00
5%,2.600000e+01
10%,2.900000e+01
25%,3.200000e+01
50%,3.600000e+01
75%,4.300000e+01
90%,4.800000e+01


In [8]:
df_framerates = pd.read_csv(rf'{export}times.csv', sep= ";")
display(df_framerates)

,Unnamed: 0,MappingMethod,FrameNumber_Start,FrameNumber_Finish,Block,Task,Trial,NumLayers,Target,Target_Relative,DateTime_Start,DateTime_Finish,Duration,NumFrames,Result,Proband,FrameRate_1
0,0,direct,65,242,1,1,0,12,3,0.250000,2021-06-29T07:19:27.465Z,2021-06-29T07:19:34.185Z,6.720,177,COMPLETED,23,26.339286
1,1,direct,295,547,1,1,1,12,11,0.916667,2021-06-29T07:19:36.096Z,2021-06-29T07:19:45.693Z,9.597,252,COMPLETED,23,26.258206
2,2,direct,604,808,1,1,2,12,9,0.750000,2021-06-29T07:19:47.749Z,2021-06-29T07:19:55.418Z,7.669,204,COMPLETED,23,26.600600
3,3,direct,849,1050,1,1,3,12,6,0.500000,2021-06-29T07:19:56.961Z,2021-06-29T07:20:04.496Z,7.535,201,TERMINATED,23,26.675514
4,4,direct,1114,1350,1,1,4,12,1,0.083333,2021-06-29T07:20:06.928Z,2021-06-29T07:20:16.018Z,9.090,236,COMPLETED,23,25.962596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6205,6205,widening,88872,89061,3,18,0,15,8,0.533333,2021-05-31T10:18:07.589Z,2021-05-31T10:18:14.593Z,7.004,189,COMPLETED,12,26.984580
6206,6206,widening,89096,89278,3,18,1,15,11,0.733333,2021-05-31T10:18:15.806Z,2021-05-31T10:18:22.355Z,6.549,182,COMPLETED,12,27.790502
6207,6207,widening,89320,89518,3,18,2,15,14,0.933333,2021-05-31T10:18:23.773Z,2021-05-31T10:18:31.158Z,7.385,198,TERMINATED,12,26.811104
6208,6208,widening,89570,89735,3,18,3,15,4,0.266667,2021-05-31T10:18:32.948Z,2021-05-31T10:18:38.840Z,5.892,165,TERMINATED,12,28.004073


In [9]:
desc_framerate = pd.DataFrame(df_framerates['FrameRate_1'].describe(percentiles=percentiles)).T

computeCI(desc_framerate)

computeWhiskers(df_framerates, desc_framerate, ['FrameRate_1'])

display(desc_framerate.T)

desc_framerate.T.to_csv(rf'{export}desc-stats_framerate.csv', sep=';')

,FrameRate_1
count,6210.000000
mean,26.609964
std,0.623125
min,17.328951
5%,25.859548
10%,26.027958
25%,26.285908
50%,26.589875
75%,26.914084
90%,27.208423


In [10]:
df_framerates['FrameTime'] = 1.0 / df_framerates['FrameRate_1'] * 1000

desc_frametime = pd.DataFrame(df_framerates['FrameTime'].describe(percentiles=percentiles)).T

computeCI(desc_frametime)

computeWhiskers(df_framerates, desc_frametime, ['FrameTime'])

display(desc_frametime.T)

desc_frametime.T.to_csv(rf'{export}desc-stats_frametime.csv', sep=';')

,FrameTime
count,6210.000000
mean,37.597718
std,0.793343
min,19.123016
5%,36.492932
10%,36.753325
25%,37.155268
50%,37.608300
75%,38.043198
90%,38.420225


In [25]:
columnNames =["DateTime", "State", "mappingMethod", "TaskNo", "TargetLayers", "LayerCount", "TrialIdx", "posX", "posY", "posZ", "TimeStamp", "iteractionType", "currentLayer"]

all_files_complete = sorted(glob.glob(path + "/*.csv"))

all_files_filtered = sorted(glob.glob(rf'{export}clean/*.csv'))

times_complete = []

for filename in all_files_complete:
    df = pd.read_csv(filename, sep=";", names = columnNames)
    first = df.iloc[0]['DateTime']
    last = df.iloc[-1]['DateTime']

    times_complete.append({'Start': first, 'End': last})

df_times_complet = pd.DataFrame(times_complete)
df_times_complet['Start_DT'] = pd.to_datetime(df_times_complet['Start'])
df_times_complet['End_DT'] = pd.to_datetime(df_times_complet['End'])
df_times_complet['Duration'] = df_times_complet['End_DT'] - df_times_complet['Start_DT']
df_times_complet['Duration_Seconds'] = df_times_complet['Duration'].apply(lambda delta: delta.seconds)
df_times_complet['Duration_Minutes'] = df_times_complet['Duration_Seconds'] / 60.0

df_times_complet.to_csv(rf'{export}duration_experiment_complete.csv', sep=';')

display(df_times_complet)


,Start,End,Start_DT,End_DT,Duration,Duration_Seconds,Duration_Minutes
0,2021-05-10T12:08:25.383Z,2021-05-10T13:02:17.032Z,2021-05-10 12:08:25.383000+00:00,2021-05-10 13:02:17.032000+00:00,0 days 00:53:51.649000,3231,53.850000
1,2021-05-11T12:21:20.944Z,2021-05-11T13:49:51.568Z,2021-05-11 12:21:20.944000+00:00,2021-05-11 13:49:51.568000+00:00,0 days 01:28:30.624000,5310,88.500000
2,2021-05-11T14:30:14.748Z,2021-05-11T15:39:21.569Z,2021-05-11 14:30:14.748000+00:00,2021-05-11 15:39:21.569000+00:00,0 days 01:09:06.821000,4146,69.100000
3,2021-05-19T12:06:30.391Z,2021-05-19T13:34:22.010Z,2021-05-19 12:06:30.391000+00:00,2021-05-19 13:34:22.010000+00:00,0 days 01:27:51.619000,5271,87.850000
4,2021-05-21T07:34:03.491Z,2021-05-21T08:47:10.600Z,2021-05-21 07:34:03.491000+00:00,2021-05-21 08:47:10.600000+00:00,0 days 01:13:07.109000,4387,73.116667
5,2021-05-21T09:24:17.038Z,2021-05-21T10:42:03.492Z,2021-05-21 09:24:17.038000+00:00,2021-05-21 10:42:03.492000+00:00,0 days 01:17:46.454000,4666,77.766667
6,2021-05-21T12:42:55.681Z,2021-05-21T14:24:10.488Z,2021-05-21 12:42:55.681000+00:00,2021-05-21 14:24:10.488000+00:00,0 days 01:41:14.807000,6074,101.233333
7,2021-05-25T12:12:50.789Z,2021-05-25T14:07:00.155Z,2021-05-25 12:12:50.789000+00:00,2021-05-25 14:07:00.155000+00:00,0 days 01:54:09.366000,6849,114.150000
8,2021-05-25T14:30:44.710Z,2021-05-25T15:45:11.829Z,2021-05-25 14:30:44.710000+00:00,2021-05-25 15:45:11.829000+00:00,0 days 01:14:27.119000,4467,74.450000
9,2021-05-27T09:18:47.999Z,2021-05-27T10:31:30.525Z,2021-05-27 09:18:47.999000+00:00,2021-05-27 10:31:30.525000+00:00,0 days 01:12:42.526000,4362,72.700000


In [24]:
times_filtered = []

for filename in all_files_filtered:
    df = pd.read_csv(filename, sep=";")
    first = df.iloc[0]['DateTime']
    last = df.iloc[-1]['DateTime']

    times_filtered.append({'Start': first, 'End': last})

df_times_filtered = pd.DataFrame(times_filtered)

display(df_times_filtered)

df_times_filtered['Start_DT'] = pd.to_datetime(df_times_filtered['Start'])
df_times_filtered['End_DT'] = pd.to_datetime(df_times_filtered['End'])
df_times_filtered['Duration'] = df_times_filtered['End_DT'] - df_times_filtered['Start_DT']
df_times_filtered['Duration_Seconds'] = df_times_filtered['Duration'].apply(lambda delta: delta.seconds)
df_times_filtered['Duration_Minutes'] = df_times_filtered['Duration_Seconds'] / 60.0

display(df_times_filtered)

df_times_filtered.to_csv(rf'{export}duration_experiment_filtered.csv', sep=';')

,Start,End
0,2021-05-10T12:13:51.726Z,2021-05-10T13:01:50.970Z
1,2021-05-11T12:28:01.814Z,2021-05-11T13:49:23.641Z
2,2021-05-11T14:34:04.393Z,2021-05-11T15:38:05.154Z
3,2021-05-19T12:16:43.886Z,2021-05-19T13:31:27.004Z
4,2021-05-21T07:40:14.494Z,2021-05-21T08:30:45.665Z
5,2021-05-21T09:29:09.365Z,2021-05-21T10:41:39.925Z
6,2021-05-21T12:50:07.258Z,2021-05-21T14:23:47.506Z
7,2021-05-25T12:31:20.212Z,2021-05-25T13:59:40.466Z
8,2021-05-25T14:38:23.245Z,2021-05-25T15:43:12.278Z
9,2021-05-27T09:32:18.934Z,2021-05-27T10:26:36.118Z


,Start,End,Start_DT,End_DT,Duration,Duration_Seconds,Duration_Minutes
0,2021-05-10T12:13:51.726Z,2021-05-10T13:01:50.970Z,2021-05-10 12:13:51.726000+00:00,2021-05-10 13:01:50.970000+00:00,0 days 00:47:59.244000,2879,47.983333
1,2021-05-11T12:28:01.814Z,2021-05-11T13:49:23.641Z,2021-05-11 12:28:01.814000+00:00,2021-05-11 13:49:23.641000+00:00,0 days 01:21:21.827000,4881,81.350000
2,2021-05-11T14:34:04.393Z,2021-05-11T15:38:05.154Z,2021-05-11 14:34:04.393000+00:00,2021-05-11 15:38:05.154000+00:00,0 days 01:04:00.761000,3840,64.000000
3,2021-05-19T12:16:43.886Z,2021-05-19T13:31:27.004Z,2021-05-19 12:16:43.886000+00:00,2021-05-19 13:31:27.004000+00:00,0 days 01:14:43.118000,4483,74.716667
4,2021-05-21T07:40:14.494Z,2021-05-21T08:30:45.665Z,2021-05-21 07:40:14.494000+00:00,2021-05-21 08:30:45.665000+00:00,0 days 00:50:31.171000,3031,50.516667
5,2021-05-21T09:29:09.365Z,2021-05-21T10:41:39.925Z,2021-05-21 09:29:09.365000+00:00,2021-05-21 10:41:39.925000+00:00,0 days 01:12:30.560000,4350,72.500000
6,2021-05-21T12:50:07.258Z,2021-05-21T14:23:47.506Z,2021-05-21 12:50:07.258000+00:00,2021-05-21 14:23:47.506000+00:00,0 days 01:33:40.248000,5620,93.666667
7,2021-05-25T12:31:20.212Z,2021-05-25T13:59:40.466Z,2021-05-25 12:31:20.212000+00:00,2021-05-25 13:59:40.466000+00:00,0 days 01:28:20.254000,5300,88.333333
8,2021-05-25T14:38:23.245Z,2021-05-25T15:43:12.278Z,2021-05-25 14:38:23.245000+00:00,2021-05-25 15:43:12.278000+00:00,0 days 01:04:49.033000,3889,64.816667
9,2021-05-27T09:32:18.934Z,2021-05-27T10:26:36.118Z,2021-05-27 09:32:18.934000+00:00,2021-05-27 10:26:36.118000+00:00,0 days 00:54:17.184000,3257,54.283333


In [27]:
desc_experiment_duration = pd.DataFrame(df_times_complet['Duration_Minutes'].describe(percentiles=percentiles)).T

computeCI(desc_experiment_duration)
computeWhiskers(df_times_complet, desc_experiment_duration, ['Duration_Minutes'])

display(desc_experiment_duration.T)

desc_experiment_duration.T.to_csv(rf'{export}desc_stats__duration_experiment-complete.csv', sep=';')

,Duration_Minutes
count,23.000000
mean,74.374638
std,16.996232
min,53.850000
5%,57.063333
10%,58.023333
25%,63.091667
50%,69.100000
75%,78.741667
90%,98.686667


In [ ]:
desc_experiment_duration = pd.DataFrame(df_times_filtered['Duration_Minutes'].describe(percentiles=percentiles)).T

computeCI(desc_experiment_duration)
computeWhiskers(df_times_filtered, desc_experiment_duration, ['Duration_Minutes'])

display(desc_experiment_duration.T)

desc_experiment_duration.T.to_csv(rf'{export}desc_stats__duration_experiment-filtered.csv', sep=';')

,Duration_Minutes
count,23.000000
mean,64.153623
std,13.080401
min,47.983333
5%,48.621667
10%,49.996667
25%,54.941667
50%,60.466667
75%,73.116667
90%,82.776667
